# 03 · Evaluate all models: FF++ (in-dataset) vs. Celeb-DF (cross-dataset)

Every trained model is evaluated **independently and identically** on:
- **FF++ test**: same kind of data it was trained on (held-out people)
- **Celeb-DF test**: an **unseen dataset** (different generator, people and quality). This measures generalisation.

Before anything is evaluated, the face crops are **audited for data leakage**.

For each model: AUC with **95% confidence intervals** (bootstrap over *videos*, since frames of one video aren't
independent), accuracy and F1 at the threshold tuned on that model's own FF++ validation set (never re-tuned on a test
set), video-level AUC, AUC per manipulation method, confusion matrices, ROC curves, its most confident mistakes, and
**parameters, GFLOPs, speed and training time**. No model is treated as the reference: significance tests are run
for **every pair** of models.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** **01_extract_faces** (FF++), **01b_extract_celebdf**, and **02_train** (plus any copies of 02)

**Produces:** `/kaggle/working/outputs/compare/` (CSV tables + figures).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
from src.kaggle_utils import setup, find_datasets, checkpoint_roots
setup(data=False)

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as Show
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve

DATASETS = find_datasets()
ROOTS = checkpoint_roots()
print("datasets:", DATASETS)
print("checkpoint folders:", ROOTS)
assert "FF++" in DATASETS, "Add the output of 01_extract_faces as an input"
assert ROOTS, "Add the output of 02_train as an input"
if "CelebDF" not in DATASETS:
    print("WARNING: Celeb-DF crops not attached; only the FF++ evaluation will run.")

## Data-leakage audit
Runs on the actual face crops before any model is evaluated. **FAIL** = leakage (stop and fix before trusting results); **WARN** = worth a look.
- every image on disk is in the manifest (no strays from earlier runs)
- no video and no FF++ identity appears in more than one split
- no identical image in two splits or in both FF++ and Celeb-DF; near-duplicates via perceptual hash
- Celeb-DF is test-only; class balance per split

In [ ]:
args = f'--data "{DATASETS["FF++"]}" --workers 4'
if "CelebDF" in DATASETS: args += f' --celebdf "{DATASETS["CelebDF"]}"'
!TQDM_MININTERVAL=60 python -m src.classifier.check_leakage {args}

## Run the evaluation
Each model is run on each test set; then the bootstrap CIs and pairwise tests are computed. About 1–2 minutes per model per dataset.

In [ ]:
roots = " ".join(f'"{r}"' for r in ROOTS)
datasets = " ".join(f'"{k}={v}"' for k, v in DATASETS.items() if k in ("FF++", "CelebDF"))
!TQDM_MININTERVAL=60 python -m src.classifier.compare --roots {roots} --datasets {datasets} --bootstrap 1000 --workers 4

OUT = "/kaggle/working/outputs/compare"
DS = [d for d in ("FF++", "CelebDF") if os.path.isdir(f"{OUT}/{d}")]
comp = {d: pd.read_csv(f"{OUT}/{d}/comparison.csv").set_index("backbone") for d in DS}
preds = {d: pd.read_csv(f"{OUT}/{d}/predictions.csv") for d in DS}
per_method = {d: pd.read_csv(f"{OUT}/{d}/per_method.csv", index_col=0) for d in DS}
summary = pd.read_csv(f"{OUT}/summary.csv")

## Overview
`*_auc_ci` = 95% CI from the video bootstrap. `auc_drop` = FF++ AUC − Celeb-DF AUC (lower = generalises better). Sorted by Celeb-DF AUC.

In [ ]:
display(summary.round(3))
for fig in ("generalisation.png", "auc_vs_compute.png"):
    if os.path.exists(f"{OUT}/{fig}"): display(Show(f"{OUT}/{fig}", width=650))
for d in DS:
    display(Markdown(f"**ROC: all models on {d}**")); display(Show(f"{OUT}/{d}/roc_all.png", width=450))

## Pairwise significance
Paired video bootstrap for every pair of models. `auc_diff_a_minus_b` > 0 means model A scored higher; the difference is significant when the 95% CI excludes 0 (p < 0.05).

In [ ]:
for d in DS:
    sig = pd.read_csv(f"{OUT}/{d}/significance.csv")
    sig["significant"] = sig.p_value < 0.05
    display(Markdown(f"### {d}"))
    display(sig.sort_values("p_value").round(4))
    mat = sig.pivot(index="model_a", columns="model_b", values="p_value")
    display(Markdown("p-value matrix")); display(mat.round(3))

## Independent evaluation of each model
Same report for every model: metrics on both test sets, AUC per manipulation, ROC and confusion matrices at the model's own FF++-validation threshold, and its most confident mistakes on the unseen Celeb-DF set.

In [ ]:
METRICS = ["auc", "auc_ci_low", "auc_ci_high", "accuracy", "f1", "video_auc", "threshold"]

def image_path(dataset, label, file):
    return f"{DATASETS[dataset]}/test/{'fake' if label == 1 else 'real'}/{file}"

def evaluate_model(backbone):
    name = comp[DS[0]].loc[backbone, "model"]
    cost = summary.set_index("backbone").loc[backbone]
    display(Markdown(f"### {name}\n{cost.params_M:.1f} M parameters · {cost.gflops:.2f} GFLOPs · "
                     f"{cost.images_per_s:.0f} images/s · trained in {cost.train_min or float('nan'):.0f} min"))
    display(pd.DataFrame({d: comp[d].loc[backbone, METRICS] for d in DS}).T.astype(float).round(4))
    display(pd.concat({d: per_method[d][name] for d in DS}, axis=0).rename("AUC").to_frame().round(4))

    fig, axes = plt.subplots(1, 2 * len(DS), figsize=(5 * len(DS) * 2, 4))
    for i, d in enumerate(DS):
        p, thr = preds[d], comp[d].loc[backbone, "threshold"]
        fpr, tpr, _ = roc_curve(p.label, p[backbone])
        axes[2 * i].plot(fpr, tpr); axes[2 * i].plot([0, 1], [0, 1], "--", color="gray")
        axes[2 * i].set_title(f"{d} ROC (AUC {comp[d].loc[backbone, 'auc']:.3f})")
        axes[2 * i].set_xlabel("FPR"); axes[2 * i].set_ylabel("TPR")
        ConfusionMatrixDisplay(confusion_matrix(p.label, (p[backbone] > thr).astype(int)),
                               display_labels=["real", "fake"]).plot(ax=axes[2 * i + 1], cmap="Blues", colorbar=False)
        axes[2 * i + 1].set_title(f"{d} @ threshold {thr:.2f}")
    plt.tight_layout(); plt.show()

    d = DS[-1]  # most confident mistakes on the cross-dataset set (FF++ if Celeb-DF is missing)
    p, thr = preds[d], comp[d].loc[backbone, "threshold"]
    worst = pd.concat([p[(p.label == 0) & (p[backbone] > thr)].nlargest(5, backbone),
                       p[(p.label == 1) & (p[backbone] <= thr)].nsmallest(5, backbone)])
    if len(worst):
        fig, axes = plt.subplots(1, len(worst), figsize=(2.6 * len(worst), 3))
        for ax, r in zip(np.atleast_1d(axes), worst.itertuples()):
            ax.imshow(Image.open(image_path(d, r.label, r.file))); ax.axis("off")
            ax.set_title(f"{'real→fake' if r.label == 0 else 'fake→real'}\nP(fake)={getattr(r, backbone):.2f}", fontsize=8)
        plt.suptitle(f"{name}: most confident mistakes on {d}", fontsize=10); plt.tight_layout(); plt.show()

for backbone in summary.backbone:
    evaluate_model(backbone)